# NB04 — Conversion & Simulation Success Rate

Runs **all 25 built-in example circuits** through the full pipeline:
1. `POST /api/converter/convert` — framework → OpenQASM 3.0
2. `POST /api/simulator/simulate` — QASM → statevector simulation

Produces a pass/fail report, framework-level success rates, and category analysis.

**Requires:** Backend running at `http://localhost:8000`  
**Output:** `../results/success_rate/conversion_report.csv`, `../results/success_rate/simulation_report.csv`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'scripts'))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

from api_timing import get_auth_token, auth_headers
from batch_converter import run_all_examples, load_fixtures

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130})

RESULTS_DIR = Path('../results/success_rate')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('✓ Imports OK')

In [ ]:
token = get_auth_token()
hdrs = auth_headers(token)
print(f'Auth: {"✓" if token else "⚠ none"}')

fixtures = load_fixtures()
print(f'Loaded {len(fixtures)} example circuits from fixtures')

## 1 — Run Full Pipeline for All Examples

This cell will take a few minutes. Each example is:
1. Sent to `/api/converter/convert`
2. The resulting QASM is sent to `/api/simulator/simulate`

In [ ]:
results = run_all_examples(
    headers=hdrs,
    simulate=True,
    sim_backend='statevector',
    sim_shots=256,
    verbose=True,
)
df = pd.DataFrame(results)
df.to_csv(RESULTS_DIR / 'pipeline_results.csv', index=False)
print(f'\n✓ Saved pipeline_results.csv ({len(df)} rows)')
df

## 2 — Overall Success Rate Summary

In [ ]:
total = len(df)
conv_pass = df['convert_success'].sum()
sim_pass = df['simulate_success'].dropna().sum()
e2e_pass = df['e2e_success'].sum()

print('=' * 50)
print(f'OVERALL SUCCESS RATES ({total} examples)')
print('=' * 50)
print(f'  Conversion success:  {conv_pass}/{total}  ({conv_pass/total:.1%})')
print(f'  Simulation success:  {int(sim_pass)}/{conv_pass}  ({sim_pass/conv_pass:.1%} of converted)')
print(f'  End-to-end success:  {e2e_pass}/{total}  ({e2e_pass/total:.1%})')
print('=' * 50)

if e2e_pass == total:
    print('\n🎉 All examples passed end-to-end!')
else:
    failed = df[~df['e2e_success']]
    print(f'\n⚠  {total - e2e_pass} example(s) failed:')
    for _, row in failed.iterrows():
        reason = row['convert_error'] if not row['convert_success'] else row['simulate_error']
        print(f'  ✗ [{row["framework"]}] {row["name"]}: {str(reason)[:100]}')

## 3 — Success Rate by Framework

In [ ]:
fw_summary = df.groupby('framework').agg(
    total=('id', 'count'),
    convert_pass=('convert_success', 'sum'),
    e2e_pass=('e2e_success', 'sum'),
    mean_convert_ms=('convert_latency_ms', 'mean'),
    mean_simulate_ms=('simulate_latency_ms', 'mean'),
).reset_index()
fw_summary['convert_rate'] = fw_summary['convert_pass'] / fw_summary['total']
fw_summary['e2e_rate'] = fw_summary['e2e_pass'] / fw_summary['total']

conversion_csv = df[['id', 'name', 'framework', 'category', 'convert_success', 'convert_latency_ms', 'convert_error']].copy()
simulation_csv = df[['id', 'name', 'framework', 'category', 'simulate_success', 'simulate_latency_ms', 'simulate_error']].copy()
conversion_csv.to_csv(RESULTS_DIR / 'conversion_report.csv', index=False)
simulation_csv.to_csv(RESULTS_DIR / 'simulation_report.csv', index=False)

print('Framework-level summary:')
display(fw_summary.round(3))
print('\n✓ Saved conversion_report.csv, simulation_report.csv')

## 4 — Plots: Success Rate & Timing

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
palette = {'cirq': '#4472C4', 'qiskit': '#ED7D31', 'pennylane': '#70AD47'}

# Success rate bars
fw_labels = fw_summary['framework'].tolist()
x = range(len(fw_labels))
colors = [palette.get(fw, 'slategray') for fw in fw_labels]

axes[0].bar(x, fw_summary['e2e_rate'] * 100, color=colors, alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels([fw.capitalize() for fw in fw_labels])
axes[0].set_ylabel('End-to-End Success Rate (%)')
axes[0].set_title('E2E Success Rate by Framework')
axes[0].set_ylim(0, 110)
axes[0].axhline(100, color='green', linestyle='--', linewidth=1)
for i, rate in enumerate(fw_summary['e2e_rate']):
    axes[0].text(i, rate * 100 + 2, f'{rate:.0%}', ha='center', fontweight='bold')

# Conversion timing
axes[1].bar(x, fw_summary['mean_convert_ms'], color=colors, alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels([fw.capitalize() for fw in fw_labels])
axes[1].set_ylabel('Mean Conversion Time (ms)')
axes[1].set_title('Avg Conversion Time by Framework')

# Simulation timing
axes[2].bar(x, fw_summary['mean_simulate_ms'].fillna(0), color=colors, alpha=0.85)
axes[2].set_xticks(x)
axes[2].set_xticklabels([fw.capitalize() for fw in fw_labels])
axes[2].set_ylabel('Mean Simulation Time (ms)')
axes[2].set_title('Avg Simulation Time by Framework')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'success_rate_by_framework.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved success_rate_by_framework.png')

## 5 — Category-Level Heatmap

In [ ]:
cat_fw = df.groupby(['category', 'framework'])['e2e_success'].mean().unstack(fill_value=float('nan'))

fig, ax = plt.subplots(figsize=(10, max(4, len(cat_fw) * 0.6 + 2)))
sns.heatmap(
    cat_fw, annot=True, fmt='.0%', cmap='RdYlGn',
    vmin=0, vmax=1, ax=ax,
    linewidths=0.5, linecolor='gray',
    cbar_kws={'label': 'E2E Success Rate'},
)
ax.set_title('E2E Success Rate: Category × Framework')
ax.set_xlabel('Framework')
ax.set_ylabel('Category')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'success_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved success_heatmap.png')

## 6 — Detailed Pass/Fail Table

In [ ]:
display_cols = ['framework', 'name', 'category', 'convert_success', 'simulate_success', 'e2e_success',
                'convert_latency_ms', 'simulate_latency_ms']
detail = df[display_cols].copy()
detail['convert_latency_ms'] = detail['convert_latency_ms'].round(0).astype('Int64')
detail['simulate_latency_ms'] = detail['simulate_latency_ms'].round(0).astype('Int64')
detail['status'] = detail['e2e_success'].map({True: '✓ PASS', False: '✗ FAIL'})
detail = detail.rename(columns={
    'framework': 'Framework', 'name': 'Example', 'category': 'Category',
    'convert_success': 'Conv ✓', 'simulate_success': 'Sim ✓', 'e2e_success': 'E2E ✓',
    'convert_latency_ms': 'Conv (ms)', 'simulate_latency_ms': 'Sim (ms)', 'status': 'Status'
})
display(detail)